# 📘 Phần 3: Xử Lý Kiểu Dữ Liệu & Ép Kiểu (Data Types & Type Conversion in Pandas)

Khi nhập dữ liệu từ file CSV, Excel hoặc Database, Pandas thường tự động suy đoán kiểu dữ liệu (`dtypes`). Tuy nhiên, do dữ liệu thực tế chứa các ký tự đặc biệt (dấu `$`, `%`, dấu phẩy, ngày tháng nhiều định dạng), các cột số và thời gian thường bị nhận diện nhầm thành kiểu `object` (chuỗi), gây khó khăn cho việc tính toán.

---

## 🎯 Mục Tiêu Bài Học:
1. Kiểm tra và phân loại kiểu dữ liệu (`dtypes`, `info()`, `select_dtypes()`).
2. Chuyển đổi dữ liệu số an toàn với `pd.to_numeric(errors='coerce')`, xử lý ký tự tiền tệ và phần trăm.
3. Chuyển đổi và trích xuất đặc trưng thời gian (`pd.to_datetime()`, `dt.year`, `dt.day_name()`, ...).
4. Tối ưu hóa bộ nhớ với kiểu dữ liệu `category`.
5. Xử lý kiểu số nguyên có chứa giá trị khuyết thiếu (`Int64` Nullable Integer).
6. Bảo toàn các trường mã định danh/số điện thoại (tránh mất số `0` ở đầu).


In [ ]:
import pandas as pd
import numpy as np

# 1. Đọc dữ liệu
df = pd.read_csv("data/customer_orders_raw.csv")
# Loại bỏ duplicate để dữ liệu gọn gàng
df = df.drop_duplicates(subset=['order_id'], keep='first').reset_index(drop=True)
print("Thông tin kiểu dữ liệu ban đầu:")
df.info()


## 1. Kiểm Tra Bộ Nhớ & Kiểu Dữ Liệu
Hãy quan sát dung lượng RAM mà DataFrame đang chiếm dụng bằng `memory_usage(deep=True)`.


In [ ]:
print("Dung lượng bộ nhớ từng cột:")
print(df.memory_usage(deep=True))
print(f"Tổng dung lượng bộ nhớ: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")


## 2. Làm Sạch & Chuyển Đổi Dữ Liệu Số (Numeric Conversion)
Cột `price` đang chứa dấu `$` và dấu phẩy `','`, và có cả chuỗi lỗi `'invalid_price'`.
Cột `discount_rate` chứa dấu `%` và các giá trị như `'None'`, `'?'`.
Cột `quantity` bị lẫn chuỗi số và số âm.

👉 **Các bước xử lý:**
1. Dùng `.str.replace()` để loại bỏ ký tự không phải số.
2. Dùng `pd.to_numeric(..., errors='coerce')` để tự động chuyển các chuỗi không thể parse thành `np.nan` thay vì làm chương trình bị crash.


In [ ]:
# 1. Xử lý cột price
# Loại bỏ ký tự '$' và ','
df['price_clean'] = df['price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()
# Chuyển đổi sang float, các giá trị lỗi sẽ thành NaN
df['price_num'] = pd.to_numeric(df['price_clean'], errors='coerce')

# 2. Xử lý cột discount_rate
df['discount_clean'] = df['discount_rate'].astype(str).str.replace('%', '', regex=False).str.strip()
df['discount_num'] = pd.to_numeric(df['discount_clean'], errors='coerce') / 100.0  # Chuyển về dạng thập phân (0.1 = 10%)
df['discount_num'] = df['discount_num'].fillna(0.0)

# 3. Xử lý cột quantity
df['quantity_num'] = pd.to_numeric(df['quantity'], errors='coerce')

df[['order_id', 'price', 'price_num', 'discount_rate', 'discount_num', 'quantity', 'quantity_num']]


## 3. Chuyển Đổi Ngày Tháng (Datetime) & Trích Xuất Đặc Trưng
Cột `order_date` chứa nhiều định dạng khác nhau: `'2023-01-15'`, `'15/01/2023'`, `'2023.01.16'`, `'invalid_date'`.

Chúng ta sử dụng `pd.to_datetime(format='mixed', errors='coerce')` để xử lý linh hoạt.


In [ ]:
# Chuyển đổi sang datetime chuẩn
df['order_datetime'] = pd.to_datetime(df['order_date'], format='mixed', errors='coerce')

# Trích xuất các thuộc tính thời gian
df['order_year'] = df['order_datetime'].dt.year
df['order_month'] = df['order_datetime'].dt.month
df['order_day'] = df['order_datetime'].dt.day
df['order_day_name'] = df['order_datetime'].dt.day_name()
df['is_weekend'] = df['order_datetime'].dt.dayofweek.isin([5, 6]).astype(int)

df[['order_id', 'order_date', 'order_datetime', 'order_year', 'order_month', 'order_day_name', 'is_weekend']]


## 4. Tối Ưu Hóa Với Kiểu `category`
Cột `product_category` và `payment_status` chỉ có một số giá trị hữu hạn lặp đi lặp lại.
Chuyển từ `object` sang `category` giúp:
1. Giảm 80-90% dung lượng bộ nhớ.
2. Tăng tốc độ lọc, groupby và sắp xếp.


In [ ]:
# Chuẩn hóa văn bản trước khi chuyển category
df['product_category_clean'] = df['product_category'].astype(str).str.strip().str.title()
df['payment_status_clean'] = df['payment_status'].astype(str).str.strip().str.upper()

# Chuyển đổi sang category
df['product_category_cat'] = df['product_category_clean'].astype('category')
df['payment_status_cat'] = df['payment_status_clean'].astype('category')

print("Kiểu category cho payment_status:")
print(df['payment_status_cat'].dtype)
print("Danh sách các nhóm (categories):", df['payment_status_cat'].cat.categories.tolist())


## 5. Xử Lý Kiểu Số Nguyên Chứa NaN (`Int64` Nullable Integer)
Trong Pandas cũ, cột số nguyên nếu có NaN sẽ bị ép sang `float64` (`28.0`).
Pandas hiện đại hỗ trợ kiểu `Int64` (chữ I viết hoa), cho phép lưu số nguyên kể cả khi có giá trị `NaN`.


In [ ]:
# Ép kiểu age sang Int64 (Nullable Integer)
df['age_num'] = pd.to_numeric(df['age'], errors='coerce')
df['age_int64'] = df['age_num'].astype('Int64')

df[['order_id', 'customer_name', 'age', 'age_int64']]


## 6. Bảo Toàn Số Điện Thoại & Mã Định Danh Dạng Chuỗi
⚠️ **Cảnh báo thường gặp:** Khi đọc file CSV, số điện thoại (`0901234567`) hoặc mã bưu điện (`01000`) rất dễ bị Pandas hiểu nhầm là số nguyên và tự động cắt bỏ số `0` ở đầu (`901234567`).

👉 **Cách khắc phục:** Luôn ép kiểu `str` hoặc dùng `dtype={'phone': str}` khi đọc file, sau đó chuẩn hóa độ dài bằng `.str.zfill()`.


In [ ]:
# Chuẩn hóa số điện thoại: loại bỏ ký tự không phải số, giữ đủ 10 chữ số
df['phone_clean'] = df['phone'].astype(str).str.replace(r'[^0-9]', '', regex=True)
# Thêm số 0 vào đầu nếu bị thiếu do lỗi đọc số
df['phone_clean'] = df['phone_clean'].apply(lambda x: ('0' + x) if (len(x) == 9 and not x.startswith('0')) else x)
df['phone_clean'] = df['phone_clean'].replace(['', '0nan', 'nan'], np.nan)

df[['customer_name', 'phone', 'phone_clean']]


## 7. Tổng Kết Các Hàm Chuyển Đổi Quan Trọng
| Kiểu Dữ Liệu | Hàm / Phương Thức Khuyên Dùng | Tùy Chọn Quan Trọng |
| :--- | :--- | :--- |
| **Số thực / Số nguyên** | `pd.to_numeric()` | `errors='coerce'` |
| **Ngày tháng / Giờ** | `pd.to_datetime()` | `format='mixed'`, `errors='coerce'` |
| **Danh mục phân loại** | `.astype('category')` | Tiết kiệm RAM, groupby nhanh |
| **Số nguyên có NaN** | `.astype('Int64')` | Giữ nguyên số nguyên không biến thành `.0` |
| **Chuỗi / Text** | `.astype(str)` hoặc `.astype('string')` | Kết hợp các hàm `.str.*` |
